<a href="https://colab.research.google.com/github/Silva015/tcc-2026/blob/main/PureConvKAN_MLSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Instalações necessárias
!pip install git+https://github.com/Blealtan/efficient-kan -q
!pip install albumentations -q
!pip install convkan -q

# Clonagem dos repositórios contendo os dados e códigos base
!git clone https://gitlab.com/lisa-unb/leguwoi.git -q
!git clone https://github.com/pedrogarciafreitas/FDCPUnBGunshotDB.git -q

import os
from glob import glob
from PIL import Image
import numpy as np
import random
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Executando no dispositivo: {device.type.upper()}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
🚀 Executando no dispositivo: CUDA


In [3]:
class GunshotDistanceDataset(Dataset):
    """
    Carrega apenas imagens de ENTRADA e atribui labels de distância:
    0: Encostado, 1: Curta, 2: Distância
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        print("🔄 Mapeando imagens de distâncias na base de dados...")

        # 1. ENCOSTADO (Label 0)
        caminho_encostado = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', 'ENCOSTADO', '*.JPG')
        for path in glob(caminho_encostado):
            self.image_paths.append(path)
            self.labels.append(0)

        # 2. CURTA DISTÂNCIA (Label 1)
        caminho_curta = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', 'CURTA', '*.JPG')
        for path in glob(caminho_curta):
            self.image_paths.append(path)
            self.labels.append(1)

        # 3. À DISTÂNCIA (Label 2)
        caminho_distancia = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', 'DISTANCIA', '*.JPG')
        for path in glob(caminho_distancia):
            self.image_paths.append(path)
            self.labels.append(2)

        print(f"✅ Concluído! Total de imagens de ENTRADA encontradas: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            # O Albumentations exige arrays numpy
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented['image']

        return image, label

In [4]:
# Resolução ideal para ConvKAN sem estourar memória na GPU
IMG_SIZE = 128
BATCH_SIZE = 8

# Transformações com Albumentations
transform_treino = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

transform_teste = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Configuração de Sementes
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

dataset_base_treino = GunshotDistanceDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_treino)
dataset_base_teste  = GunshotDistanceDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_teste)

# Separação Semântica (Por Caso)
case_to_indices = {}
for idx, path in enumerate(dataset_base_treino.image_paths):
    filename = os.path.basename(path)
    case_id = filename[:9]
    if case_id not in case_to_indices:
        case_to_indices[case_id] = []
    case_to_indices[case_id].append(idx)

unique_cases = list(case_to_indices.keys())
cases_treino, cases_teste = train_test_split(unique_cases, test_size=0.2, random_state=RANDOM_SEED)

# Mapeamento
indices_treino = [idx for case in cases_treino for idx in case_to_indices[case]]
indices_teste = [idx for case in cases_teste for idx in case_to_indices[case]]

dataset_treino = Subset(dataset_base_treino, indices_treino)
dataset_teste  = Subset(dataset_base_teste, indices_teste)

trainloader = DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True)
testloader  = DataLoader(dataset_teste, batch_size=BATCH_SIZE, shuffle=False)

print("\n" + "="*50)
print("🚀 SANITY CHECK DE VAZAMENTO DE DADOS")
print("="*50)

train_cases_check = set([os.path.basename(dataset_base_treino.image_paths[i])[:9] for i in indices_treino])
test_cases_check  = set([os.path.basename(dataset_base_teste.image_paths[i])[:9] for i in indices_teste])
intersecao = train_cases_check.intersection(test_cases_check)

print(f"📊 Total de Casos Únicos: {len(unique_cases)}")
print(f"   -> Casos no Treino: {len(train_cases_check)} casos ({len(dataset_treino)} imagens)")
print(f"   -> Casos no Teste:  {len(test_cases_check)} casos ({len(dataset_teste)} imagens)")

if len(intersecao) == 0:
    print("\n✅ SUCESSO! Zero interseção semântica identificada.")
else:
    print(f"\n❌ ALERTA DE VAZAMENTO! {len(intersecao)} casos vazados.")

🔄 Mapeando imagens de distâncias na base de dados...
✅ Concluído! Total de imagens de ENTRADA encontradas: 1883
🔄 Mapeando imagens de distâncias na base de dados...
✅ Concluído! Total de imagens de ENTRADA encontradas: 1883

🚀 SANITY CHECK DE VAZAMENTO DE DADOS
📊 Total de Casos Únicos: 441
   -> Casos no Treino: 352 casos (1532 imagens)
   -> Casos no Teste:  89 casos (351 imagens)

✅ SUCESSO! Zero interseção semântica identificada.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_16204/1206918511.py:12: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
/tmp/ipykernel_16204/1206918511.py:13: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),


In [5]:
from convkan import ConvKAN

class CustomPureConvKAN_MLSD(nn.Module):
    def __init__(self):
        super(CustomPureConvKAN_MLSD, self).__init__()

        # 1º Bloco: Extração de baixo nível (Stride=2 na entrada como padrão)
        self.conv1 = ConvKAN(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn1 = nn.BatchNorm2d(16)

        # 2º Bloco: Padrões intermediários
        self.conv2 = ConvKAN(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn2 = nn.BatchNorm2d(32)

        # 3º Bloco: Padrões complexos
        self.conv3 = ConvKAN(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn3 = nn.BatchNorm2d(64)

        # Global Average Pooling para independência de resolução
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)

        # MUDANÇA AQUI: Classificação final para 3 classes de MLSD
        self.fc = nn.Linear(64, 3)

    def forward(self, x):
        x = self.bn1(self.pool1(self.conv1(x)))
        x = self.bn2(self.pool2(self.conv2(x)))
        x = self.bn3(self.pool3(self.conv3(x)))

        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        x = self.fc(x)
        return x

model_mlsd = CustomPureConvKAN_MLSD().to(device)
print(f"🚀 Modelo CustomPureConvKAN_MLSD inicializado e enviado para: {device.type.upper()}")

🚀 Modelo CustomPureConvKAN_MLSD inicializado e enviado para: CUDA


In [6]:
total_epochs = 60

# 1. Cálculo dos pesos (devido ao desbalanceamento: Encostado vs Curta vs Distância)
train_labels = [dataset_base_treino.labels[i] for i in indices_treino]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"⚖️ Pesos das classes [Encostado, Curta, Distância]: {class_weights_tensor.cpu().numpy()}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

best_acc = 0.0
best_model_wts = copy.deepcopy(model_mlsd.state_dict())

optimizer = torch.optim.AdamW(model_mlsd.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

print(f"🔥 Iniciando Treinamento da PureConvKAN para MLSD ({total_epochs} épocas)...")

for epoch in range(total_epochs):
    # FASE DE TREINO
    model_mlsd.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_mlsd(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        running_loss += loss.item()

    acc_train = 100 * correct_train / total_train

    # FASE DE TESTE
    model_mlsd.eval()
    correct_test, total_test = 0, 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_mlsd(images)

            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    acc_test = 100 * correct_test / total_test
    scheduler.step()

    # CHECKPOINT
    if acc_test > best_acc:
        best_acc = acc_test
        best_model_wts = copy.deepcopy(model_mlsd.state_dict())
        torch.save(best_model_wts, 'melhor_modelo_distancia_pureconvkan.pth')
        indicador_recorde = "🏆 NOVO RECORDE!"
    else:
        indicador_recorde = ""

    print(f"Época {epoch+1:03d}/{total_epochs} | Treino: {acc_train:.2f}% (Loss: {running_loss/len(trainloader):.4f}) | Teste: {acc_test:.2f}% {indicador_recorde}")

print(f"\n✅ Treinamento concluído. Melhor Acurácia de Validação: {best_acc:.2f}%")

⚖️ Pesos das classes [Encostado, Curta, Distância]: [15.474748    2.9689922   0.38482794]
🔥 Iniciando Treinamento da PureConvKAN para MLSD (60 épocas)...
Época 001/60 | Treino: 47.00% (Loss: 1.1487) | Teste: 74.36% 🏆 NOVO RECORDE!
Época 002/60 | Treino: 69.39% (Loss: 1.0380) | Teste: 80.34% 🏆 NOVO RECORDE!
Época 003/60 | Treino: 73.43% (Loss: 1.0262) | Teste: 79.49% 
Época 004/60 | Treino: 75.07% (Loss: 0.9889) | Teste: 78.06% 
Época 005/60 | Treino: 77.15% (Loss: 0.9758) | Teste: 80.06% 
Época 006/60 | Treino: 78.00% (Loss: 0.9541) | Teste: 76.07% 
Época 007/60 | Treino: 77.68% (Loss: 0.9157) | Teste: 77.21% 
Época 008/60 | Treino: 77.42% (Loss: 0.9374) | Teste: 78.92% 
Época 009/60 | Treino: 79.05% (Loss: 0.9153) | Teste: 79.49% 
Época 010/60 | Treino: 78.00% (Loss: 0.9142) | Teste: 76.92% 
Época 011/60 | Treino: 77.81% (Loss: 0.9418) | Teste: 80.34% 
Época 012/60 | Treino: 80.09% (Loss: 0.9326) | Teste: 76.92% 
Época 013/60 | Treino: 79.05% (Loss: 0.9033) | Teste: 73.22% 
Época 014/

In [7]:
print("📥 Carregando o estado ótimo do modelo para geração de métricas...")
model_mlsd.load_state_dict(torch.load('melhor_modelo_distancia_pureconvkan.pth'))
model_mlsd.eval()

y_true = []
y_pred = []
y_prob_both = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        outputs = model_mlsd(images)

        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_prob_both.extend(probabilities.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob_both = np.array(y_prob_both)

# Binarização necessária para o cálculo do AUC Multiclasse (OVR)
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])

classes = ['Encostado (0)', 'Curta (1)', 'Distância (2)']

cm = confusion_matrix(y_true, y_pred)

FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)
support_per_class = cm.sum(axis=1)

acc = accuracy_score(y_true, y_pred)
metrics = {}
averages = ['macro', 'micro', 'weighted']

for avg in averages:
    metrics[f'Prec_{avg}'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'Rec_{avg}'] = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'F1_{avg}'] = f1_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'AUC_{avg}'] = roc_auc_score(y_true_bin, y_prob_both, multi_class='ovr', average=avg)

specificity_per_class = np.divide(TN, (TN + FP), out=np.zeros_like(TN, dtype=float), where=(TN + FP)!=0)

metrics['Spec_macro'] = np.mean(specificity_per_class)
metrics['Spec_micro'] = TN.sum() / (TN.sum() + FP.sum()) if (TN.sum() + FP.sum()) > 0 else 0
metrics['Spec_weighted'] = np.average(specificity_per_class, weights=support_per_class)

# ==========================================
# GERAÇÃO DA TABELA DO ARTIGO (Console)
# ==========================================
print("\n" + "="*145)
print(" TABELA 5 - PERFORMANCE METRICS MLSD (FORMATO CIENTÍFICO)")
print("="*145)

header1 = f"{'Architecture':<14} {'Variant':<10} {'ACC':<7} "
header1 += f"{'Precision':<23} {'Recall':<23} {'F1-Score':<23} {'AUC':<23} {'Specificity':<23}"
print(header1)

header2 = f"{'':<33} "
for _ in range(5):
    header2 += f"{'M':<7} {'m':<7} {'W':<7}   "
print(header2)
print("-" * 145)

row = f"{'PureConvKAN':<14} {'MLSD':<10} {acc:<7.3f} "
row += f"{metrics['Prec_macro']:<7.3f} {metrics['Prec_micro']:<7.3f} {metrics['Prec_weighted']:<7.3f}   "
row += f"{metrics['Rec_macro']:<7.3f} {metrics['Rec_micro']:<7.3f} {metrics['Rec_weighted']:<7.3f}   "
row += f"{metrics['F1_macro']:<7.3f} {metrics['F1_micro']:<7.3f} {metrics['F1_weighted']:<7.3f}   "
row += f"{metrics['AUC_macro']:<7.3f} {metrics['AUC_micro']:<7.3f} {metrics['AUC_weighted']:<7.3f}   "
row += f"{metrics['Spec_macro']:<7.3f} {metrics['Spec_micro']:<7.3f} {metrics['Spec_weighted']:<7.3f}"
print(row)
print("="*145 + "\n")

print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

📥 Carregando o estado ótimo do modelo para geração de métricas...

 TABELA 5 - PERFORMANCE METRICS MLSD (FORMATO CIENTÍFICO)
Architecture   Variant    ACC     Precision               Recall                  F1-Score                AUC                     Specificity            
                                  M       m       W         M       m       W         M       m       W         M       m       W         M       m       W         
-------------------------------------------------------------------------------------------------------------------------------------------------
PureConvKAN    MLSD       0.803   0.268   0.803   0.645     0.333   0.803   0.803     0.297   0.803   0.716     0.536   0.888   0.509     0.667   0.902   0.197  

               precision    recall  f1-score   support

Encostado (0)       0.00      0.00      0.00         9
    Curta (1)       0.00      0.00      0.00        60
Distância (2)       0.80      1.00      0.89       282

     accuracy            